# Movie Database MCP Server

Build and validate a high-level Model Context Protocol (MCP) server backed by the workshop movie database.

By the end of this notebook, you will be able to:

- inspect the SQLite movie data;
- create a reusable database helper that returns MCP content;
- expose the helper as a FastMCP `search_movies` tool; and
- discover and call the tool with the NeMo Agent Toolkit (`nat`) CLI.

## Setup

The notebook can be opened from either the repository root or the `notebooks/` directory. This cell moves to the repository root so the database and generated server files use stable relative paths.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "movie.sqlite").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_PATH = "data/movie.sqlite"
if not (PROJECT_ROOT / DB_PATH).is_file():
    raise FileNotFoundError(f"Could not find {DB_PATH} from {Path.cwd()}")

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")
print(f"Database: {PROJECT_ROOT / DB_PATH}")

## Preview Some Rows

Let's preview some rows from the `IMDB` table to understand the data structure. Each movie has an ID, title, rating, vote count, budget, and runtime.

In [ ]:
import json
import sqlite3

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()
cursor.execute(
    "SELECT Movie_id, Title, Rating, TotalVotes, Budget, Runtime "
    "FROM IMDB LIMIT 5"
)
rows = [dict(row) for row in cursor.fetchall()]
conn.close()

print(json.dumps(rows, indent=2, ensure_ascii=False))

## Create the MovieDB Helper

The helper owns the parameterized SQL query and converts each result into a one-element list containing `mcp.types.TextContent`. The text field contains the JSON-encoded result list, which is the content shape returned to MCP clients.

In [ ]:
%%writefile movie_db.py
"""SQLite helper used by the movie database MCP server."""

import json
import sqlite3
from pathlib import Path

import mcp.types as types


class MovieDB:
    """Read-only queries for the workshop movie database."""

    def __init__(self, db_path: str | Path):
        self.db_path = Path(db_path).expanduser().resolve()
        if not self.db_path.is_file():
            raise FileNotFoundError(f"Movie database not found: {self.db_path}")

    def _search_movies(
        self,
        title: str | None,
        min_rating: float | None,
        max_rating: float | None,
        limit: int = 20,
    ) -> list[types.TextContent]:
        """Search movies in the IMDB table by title and/or rating range."""
        if not 1 <= limit <= 100:
            raise ValueError("limit must be between 1 and 100")
        if (
            min_rating is not None
            and max_rating is not None
            and min_rating > max_rating
        ):
            raise ValueError("min_rating cannot be greater than max_rating")

        query = """
            SELECT Movie_id, Title, Rating, TotalVotes, Budget, Runtime
            FROM IMDB
        """
        params: list[str | float | int] = []
        conditions: list[str] = []

        if title is not None:
            conditions.append("Title LIKE ?")
            params.append(f"%{title}%")
        if min_rating is not None:
            conditions.append("Rating >= ?")
            params.append(min_rating)
        if max_rating is not None:
            conditions.append("Rating <= ?")
            params.append(max_rating)
        if conditions:
            query += " WHERE " + " AND ".join(conditions)

        query += " ORDER BY Rating DESC LIMIT ?"
        params.append(limit)

        with sqlite3.connect(self.db_path) as conn:
            conn.row_factory = sqlite3.Row
            rows = conn.execute(query, params).fetchall()

        output = [
            {
                "movie_id": row["Movie_id"],
                "title": row["Title"],
                "rating": row["Rating"],
                "total_votes": row["TotalVotes"],
                "budget": row["Budget"],
                "runtime": row["Runtime"],
            }
            for row in rows
        ]
        return [
            types.TextContent(
                type="text",
                text=json.dumps(output, ensure_ascii=False),
            )
        ]


In [ ]:
from movie_db import MovieDB

movie_db = MovieDB(DB_PATH)

## Test the Helper Class

Before building the MCP server, verify the `MovieDB` class with several query patterns.

Search by title — find movies with `Dark` in the title:

In [ ]:
movie_db._search_movies(title="Dark", min_rating=None, max_rating=None)

Search by minimum rating — movies rated 8.5 or higher:

In [ ]:
movie_db._search_movies(
    title=None, min_rating=8.5, max_rating=None, limit=5
)

Combined filters — movies with `The` in the title and a rating above 8.0:

In [ ]:
movie_db._search_movies(
    title="The", min_rating=8.0, max_rating=None, limit=3
)

No filters — return the top five movies by rating:

In [ ]:
movie_db._search_movies(
    title=None, min_rating=None, max_rating=None, limit=5
)

## Create the High-Level MCP Server

FastMCP turns the helper method into a discoverable `search_movies` tool and serves it with the streamable HTTP transport.

In [ ]:
%%writefile movie_server.py
"""High-level FastMCP server for the workshop movie database."""

import argparse
import os
from pathlib import Path

from mcp.server.fastmcp import FastMCP

from movie_db import MovieDB


def build_server(db_path: str | Path, host: str, port: int) -> FastMCP:
    """Build a FastMCP server exposing the movie search tool."""
    movie_db = MovieDB(db_path)
    server = FastMCP(
        "movie-database",
        instructions="Search the workshop IMDB movie database.",
        host=host,
        port=port,
    )

    @server.tool()
    def search_movies(
        title: str | None = None,
        min_rating: float | None = None,
        max_rating: float | None = None,
        limit: int = 20,
    ):
        """Search movies by title and/or IMDB rating range."""
        return movie_db._search_movies(title, min_rating, max_rating, limit)

    return server


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "db_path",
        nargs="?",
        default="data/movie.sqlite",
        help="Path to the SQLite movie database.",
    )
    args = parser.parse_args()

    host = os.environ.get("MCP_HOST", "127.0.0.1")
    port = int(os.environ["MCP_PORT"])
    server = build_server(args.db_path, host, port)
    server.run(transport="streamable-http")


if __name__ == "__main__":
    main()


## Start the MCP Server

Launch the server as a background process and give it a few seconds to start. The server uses `MCP_PORT` configured in `.vscode/notebook.env`.

In [ ]:
import subprocess
import time

MCP_PORT = int(os.environ["MCP_PORT"])

# Start the HTTP server in the background
mcp_server_process = subprocess.Popen(
    ["python", "movie_server.py", DB_PATH],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for the server to start
time.sleep(3)

SERVER_PID = mcp_server_process.pid
MCP_URL = f"http://127.0.0.1:{MCP_PORT}/mcp"
print(f"Server started on {MCP_URL} (PID: {SERVER_PID})")

## Introduction to NVIDIA NeMo Agent Toolkit

NVIDIA NeMo Agent Toolkit (NAT) is a config-driven framework for building, running, evaluating, and observing agentic workflows. A NAT workflow connects three kinds of components:

- **Functions and function groups** provide tools, including tools discovered from MCP servers.
- **LLMs** define the model provider and model configuration used for reasoning.
- **Workflows** define how an agent combines the LLM with its available tools.

This notebook uses NAT only as an MCP client to verify the server independently. Notebook 4 builds on this foundation by connecting the same `search_movies` tool to a Nemotron-powered ReAct workflow.

## Test with the NAT CLI

`nat` is the command-line entry point installed with the NeMo Agent Toolkit. Its MCP client commands let us validate the server before connecting it to a larger agent workflow.

| Command | What it does |
| --- | --- |
| `nat run --config_file workflow.yml --input "..."` | Execute a configured workflow once. |
| `nat serve --config_file workflow.yml` | Expose a workflow as an HTTP/SSE API. |
| `nat eval --config_file workflow.yml` | Evaluate a workflow against a dataset. |
| `nat mcp client tool list --url ...` | Discover MCP tools and their schemas. |
| `nat mcp client tool call <name> --url ... --json-args '{...}'` | Invoke an MCP tool and print its response. |
| `nat info components` | List available toolkit components. |

First, confirm that the server advertises `search_movies`:

In [ ]:
!nat mcp client tool list --direct --url http://127.0.0.1:$MCP_PORT/mcp

Now call `search_movies` directly and return up to five movies rated 8.5 or higher:

In [ ]:
!nat mcp client tool call search_movies --direct --url http://127.0.0.1:$MCP_PORT/mcp --json-args '{"min_rating": 8.5, "limit": 5}'

## Stop the MCP Server

Always stop background server processes when the exercise is complete.

In [ ]:
mcp_server_process.terminate()
mcp_server_process.wait()

print(f"Server (PID: {SERVER_PID}) stopped")

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.